##### As seen in the the Cleaning Phase, there is a massive 7 month gap for plant 710. I will use `Donor Imputation Method` to fill the missing values based on plants 515, which possess a similar range of `m3` per hour (rounded, e.g. 9:00 for all remissions between 9:00 and 9:59) and similar count of  remissions (rows) also per hour.

##### There is also a 1 month gap for all plants from `March 31 2025` - `May 2 2025`, for which I will use RandomForest Regressor and historic data for each plant to predict the missing data accurately.

We will analyze patterns and learn using `RandomForestRegressor` and `MultiOutputRegressor`

In [1]:
# DATA IMPUTATION FOR PLANT 710 MISSING 7 MONTH GAP

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

# Config
DATA_PATH = "../data/processed/remissions_db_cleaned.xlsx"
PLANT_TARGET = "710"
PLANT_PRIMARY = "515"
GAP_START = "2023-05-17"
GAP_END = "2024-02-02"
FALLBACK_TOP_K = 3
RANDOM_STATE = 42
# Optional: force weekend schedule for 710 (set to None to disable)
DOW_SCALE_OVERRIDE = {5: 0.7, 6: 0.15}
# Blend dow scaling toward 1.0 to avoid over-suppressing counts
DOW_BLEND = 0.6
# Use 710 vs 515 ratios for weekend scaling
USE_515_DOW_RATIOS = True
# Optional: scale per-row volume by 710 vs 515 dow ratios
APPLY_VOLUME_DOW_SCALE = False
VOLUME_DOW_BLEND = 0.5
VOLUME_DOW_CLIP = (0.6, 1.4)
# Keep real 515 volume distribution (preserves max values)
USE_515_FOR_VOLUME = True
# Exclude u_Volumen from RF so it keeps real distribution
EXCLUDE_VOLUME_FROM_RF = True
# Blend RF hourly counts toward 710 pre-gap mean by day-of-week and hour
COUNT_BLEND_WEEKDAY = 0.6
COUNT_BLEND_WEEKEND = 0.0
# Count sampling from 515 by (hour, dow, month), scaled by both ratios
USE_515_COUNT_SAMPLING = True
COUNT_SAMPLE_SCALE_HOUR = True
COUNT_SAMPLE_SCALE_DOW = True
COUNT_MAX_Q = 0.98
# Enforce inactive hours using 710 pre-gap probability
ENFORCE_ACTIVE_HOURS = True
ACTIVE_PROB_SCALE_DOW = True
ACTIVE_PROB_FLOOR = 0.0
# Inject weekday spikes from 710 pre-gap counts
USE_710_WEEKDAY_SPIKES = True
WEEKDAY_SPIKE_PROB = 0.1
WEEKDAY_SPIKE_MIN = 7
WEEKDAY_SPIKE_TAIL_POWER = 1.0
WEEKDAY_SPIKE_MAX_Q = 0.995

# Load and parse
remissions = pd.read_excel(DATA_PATH)
for col in ["order_date", "typed_time", "start_time", "at_plant_time"]:
    if col in remissions.columns:
        remissions[col] = pd.to_datetime(remissions[col], errors="coerce")

remissions["ship_plant_code"] = remissions["ship_plant_code"].astype(str).str.strip()

# Helpers
def make_time_features(dt_series):
    dt = pd.to_datetime(dt_series)
    return pd.DataFrame({
        "hour": dt.dt.hour,
        "day_of_week": dt.dt.dayofweek,
        "month": dt.dt.month,
        "day_of_year": dt.dt.dayofyear,
        "is_weekend": (dt.dt.dayofweek >= 5).astype(int),
    })

# Build donor pool based on similarity to plant 515 (pre-gap)
gap_start = pd.Timestamp(GAP_START)
ref = remissions.dropna(subset=["typed_time"]).copy()
ref = ref[ref["typed_time"] < gap_start]
ref["hour"] = ref["typed_time"].dt.hour

counts = ref.groupby(["ship_plant_code", "hour"]).size().unstack(fill_value=0)
vols = ref.groupby(["ship_plant_code", "hour"])["u_Volumen"].mean().unstack(fill_value=0)

def plant_profile(plant_code):
    if plant_code in counts.index:
        c = counts.loc[plant_code].reindex(range(24), fill_value=0)
    else:
        c = pd.Series(0, index=range(24))
    if plant_code in vols.index:
        v = vols.loc[plant_code].reindex(range(24), fill_value=0)
    else:
        v = pd.Series(0, index=range(24))
    if c.sum() > 0:
        c = c / c.sum()
    if v.sum() > 0:
        v = v / v.sum()
    return np.concatenate([c.values, v.values])

primary_profile = plant_profile(PLANT_PRIMARY)
sims = {}
for plant in counts.index:
    if plant in [PLANT_PRIMARY, PLANT_TARGET]:
        continue
    vec = plant_profile(plant)
    denom = np.linalg.norm(primary_profile) * np.linalg.norm(vec)
    sim = float(primary_profile.dot(vec) / denom) if denom > 0 else 0.0
    sims[plant] = sim

fallback_plants = [p for p, _ in sorted(sims.items(), key=lambda kv: kv[1], reverse=True)[:FALLBACK_TOP_K]]
donor_plants = [PLANT_PRIMARY] + fallback_plants
print("Donor plants:", donor_plants)

# Donor data
pool = remissions[remissions["ship_plant_code"].isin(donor_plants)].copy()
pool = pool.dropna(subset=["typed_time"])

# Pre-gap 710 baseline for scaling
p710_hist = remissions[
    (remissions["ship_plant_code"] == PLANT_TARGET) &
    (remissions["typed_time"].notna()) &
    (remissions["typed_time"] < gap_start)
] .copy()

# 710 hourly counts and spike source
p710_hourly_counts = None
p710_counts_by_dh = None
if not p710_hist.empty:
    p710_hourly = p710_hist.copy()
    p710_hourly["hour_bucket"] = p710_hourly["typed_time"].dt.floor("h")
    p710_hourly_counts = p710_hourly.groupby("hour_bucket").size().reset_index(name="count")
    p710_hourly_counts["day_of_week"] = p710_hourly_counts["hour_bucket"].dt.dayofweek
    p710_hourly_counts["hour"] = p710_hourly_counts["hour_bucket"].dt.hour
    p710_counts_by_dh = p710_hourly_counts.groupby(["day_of_week", "hour"])["count"].apply(list)

# 710 active hour probabilities (pre-gap)
prob_active = None
if not p710_hist.empty:
    p710_active = p710_hist.copy()
    p710_active["date"] = p710_active["typed_time"].dt.normalize()
    p710_active["hour"] = p710_active["typed_time"].dt.hour
    p710_active["day_of_week"] = p710_active["typed_time"].dt.dayofweek
    days_by_dow = (
        p710_active[["day_of_week", "date"]]
        .drop_duplicates()
        .groupby("day_of_week").size()
    )
    active_by_dh = (
        p710_active.groupby(["day_of_week", "hour", "date"]).size()
        .reset_index()
        .groupby(["day_of_week", "hour"]).size()
    )
    prob_active = (active_by_dh / days_by_dow).reindex(
        pd.MultiIndex.from_product([range(7), range(24)])
    )
    prob_active = prob_active.fillna(0.0).clip(0.0, 1.0)

# 515 pre-gap ratios for diagnostics and optional scaling
p515_hist = remissions[
    (remissions["ship_plant_code"] == PLANT_PRIMARY) &
    (remissions["typed_time"] .notna()) &
    (remissions["typed_time"] < gap_start)
] .copy()
ratio_515_counts = None
ratio_515_volume = None
ratio_515_hour = None
if not p710_hist.empty and not p515_hist.empty:
    p710_hist["day_of_week"] = p710_hist["typed_time"].dt.dayofweek
    p515_hist["day_of_week"] = p515_hist["typed_time"].dt.dayofweek
    ratio_515_counts = (
        p710_hist.groupby("day_of_week").size() /
        p515_hist.groupby("day_of_week").size()
    ).replace([np.inf, -np.inf], np.nan).reindex(range(7))
    if "u_Volumen" in remissions.columns:
        ratio_515_volume = (
            p710_hist.groupby("day_of_week")["u_Volumen"].sum() /
            p515_hist.groupby("day_of_week")["u_Volumen"].sum()
        ).replace([np.inf, -np.inf], np.nan).reindex(range(7))
    ratio_515_hour = (
        p710_hist.groupby(p710_hist["typed_time"].dt.hour).size() /
        p515_hist.groupby(p515_hist["typed_time"].dt.hour).size()
    ).replace([np.inf, -np.inf], np.nan).reindex(range(24))
    ratio_table = pd.DataFrame({
        "ratio_count_710_vs_515": ratio_515_counts,
        "ratio_volume_710_vs_515": ratio_515_volume,
    })
    print("710 vs 515 day-of-week ratios (pre-gap):")
    print(ratio_table)
    ratio_515_counts = ratio_515_counts.fillna(1.0).clip(0.1, 3.0)
    if ratio_515_volume is not None:
        ratio_515_volume = ratio_515_volume.fillna(1.0).clip(0.3, 2.0)
    ratio_515_hour = ratio_515_hour.fillna(1.0).clip(0.3, 3.0)

p515_hourly_counts = None
idx_cnt_full = None
idx_cnt_hour = None
if not p515_hist.empty:
    p515_hourly = p515_hist.copy()
    p515_hourly["hour_bucket"] = p515_hourly["typed_time"].dt.floor("h")
    p515_hourly_counts = p515_hourly.groupby("hour_bucket").size().reset_index(name="count")
    p515_hourly_counts["hour"] = p515_hourly_counts["hour_bucket"].dt.hour
    p515_hourly_counts["day_of_week"] = p515_hourly_counts["hour_bucket"].dt.dayofweek
    p515_hourly_counts["month"] = p515_hourly_counts["hour_bucket"].dt.month
    idx_cnt_full = p515_hourly_counts.groupby(["hour", "day_of_week", "month"]).indices
    idx_cnt_hour = p515_hourly_counts.groupby(["hour"]).indices

pool["hour"] = pool["typed_time"].dt.hour
pool["day_of_week"] = pool["typed_time"].dt.dayofweek
donor_hour_counts = pool.groupby("hour").size()
donor_dow_counts = pool.groupby("day_of_week").size()

if not p710_hist.empty:
    p710_hist["hour"] = p710_hist["typed_time"].dt.hour
    p710_hist["day_of_week"] = p710_hist["typed_time"].dt.dayofweek
    ratio_by_hour = (p710_hist.groupby("hour").size() / donor_hour_counts).replace([np.inf, -np.inf], np.nan)
    ratio_by_hour = ratio_by_hour.reindex(range(24), fill_value=np.nan)
    ratio_mean = ratio_by_hour.mean()
    ratio_by_hour = ratio_by_hour.fillna(ratio_mean if pd.notna(ratio_mean) else 1.0)
    ratio_by_hour = ratio_by_hour.clip(0.3, 3.0)
    ratio_by_dow = (p710_hist.groupby("day_of_week").size() / donor_dow_counts).replace([np.inf, -np.inf], np.nan)
    ratio_by_dow = ratio_by_dow.reindex(range(7), fill_value=np.nan)
    ratio_dow_mean = ratio_by_dow.mean()
    ratio_by_dow = ratio_by_dow.fillna(ratio_dow_mean if pd.notna(ratio_dow_mean) else 1.0)
    ratio_by_dow = ratio_by_dow.clip(0.2, 3.0)
else:
    ratio_by_hour = pd.Series(1.0, index=range(24))
    ratio_by_dow = pd.Series(1.0, index=range(7))

if USE_515_DOW_RATIOS and ratio_515_counts is not None:
    for dow in [5, 6]:
        ratio_by_dow.loc[dow] = ratio_515_counts.loc[dow]

if DOW_SCALE_OVERRIDE:
    for dow, val in DOW_SCALE_OVERRIDE.items():
        ratio_by_dow.loc[dow] = val

ratio_by_dow = ratio_by_dow.clip(0.0, 3.0)
if DOW_BLEND is not None:
    ratio_by_dow = (1 - DOW_BLEND) + DOW_BLEND * ratio_by_dow

if ENFORCE_ACTIVE_HOURS and prob_active is not None:
    if ACTIVE_PROB_SCALE_DOW:
        dow_scale = ratio_by_dow.clip(0.0, 1.0)
        prob_active = prob_active.mul(dow_scale, level=0)
    prob_active = prob_active.clip(ACTIVE_PROB_FLOOR, 1.0)

# Train hourly count model on donor pool
pool["hour_bucket"] = pool["typed_time"].dt.floor("h")
hourly_counts = pool.groupby("hour_bucket").size().reset_index(name="count")
X_counts = make_time_features(hourly_counts["hour_bucket"])
y_counts = hourly_counts["count"]

donor_hourly_counts = hourly_counts["count"]
max_count = int(np.ceil(np.quantile(donor_hourly_counts, COUNT_MAX_Q)))
if USE_515_COUNT_SAMPLING and p515_hourly_counts is not None:
    max_count = int(np.ceil(np.quantile(p515_hourly_counts["count"], COUNT_MAX_Q)))
if p710_hourly_counts is not None:
    max_count = max(max_count, int(np.ceil(np.quantile(p710_hourly_counts["count"], COUNT_MAX_Q))))
max_spike = max_count
if WEEKDAY_SPIKE_MAX_Q is not None and p710_hourly_counts is not None:
    max_spike = max(
        max_spike,
        int(np.ceil(np.quantile(p710_hourly_counts["count"], WEEKDAY_SPIKE_MAX_Q)))
    )

rf_count = RandomForestRegressor(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_count.fit(X_counts, y_counts)

# Gap hours for plant 710
raw_gap_end = pd.Timestamp(GAP_END)
gap_end = raw_gap_end + pd.Timedelta(hours=23)
full_gap_hours = pd.date_range(gap_start, gap_end, freq="h")

p710 = remissions[
    (remissions["ship_plant_code"] == PLANT_TARGET) &
    (remissions["typed_time"].notna())
]
existing_hours = p710["typed_time"].dt.floor("h").unique()
gap_hours = full_gap_hours.difference(existing_hours)

rng = np.random.default_rng(RANDOM_STATE)
if USE_515_COUNT_SAMPLING and p515_hourly_counts is not None:
    base_counts = []
    for ts in gap_hours:
        key = (ts.hour, ts.dayofweek, ts.month)
        idxs = idx_cnt_full.get(key) if idx_cnt_full is not None else None
        if idxs is None:
            idxs = idx_cnt_hour.get(ts.hour) if idx_cnt_hour is not None else None
            if idxs is None:
                idxs = np.arange(len(p515_hourly_counts))
        idx = rng.choice(idxs)
        base_counts.append(p515_hourly_counts.iloc[idx]["count"])
    pred_counts = np.array(base_counts, dtype=float)

    if COUNT_SAMPLE_SCALE_HOUR and ratio_515_hour is not None:
        hour_scale_series = ratio_515_hour
    else:
        hour_scale_series = pd.Series(1.0, index=range(24))
    scale_hour = pd.Series(gap_hours.hour).map(hour_scale_series).fillna(1.0).to_numpy()

    if COUNT_SAMPLE_SCALE_DOW and ratio_515_counts is not None:
        dow_scale_series = ratio_515_counts.copy()
    else:
        dow_scale_series = pd.Series(1.0, index=range(7))
    if DOW_SCALE_OVERRIDE:
        for dow, val in DOW_SCALE_OVERRIDE.items():
            dow_scale_series.loc[dow] = val
    if DOW_BLEND is not None:
        dow_scale_series = (1 - DOW_BLEND) + DOW_BLEND * dow_scale_series
    scale_dow = pd.Series(gap_hours.dayofweek).map(dow_scale_series).fillna(1.0).to_numpy()

    pred_counts = np.clip(np.rint(pred_counts * scale_hour * scale_dow), 0, max_count).astype(int)
else:
    X_gap = make_time_features(pd.Series(gap_hours))
    pred_counts = rf_count.predict(X_gap)
    scale_hour = pd.Series(gap_hours.hour).map(ratio_by_hour).fillna(1.0).to_numpy()
    scale_dow = pd.Series(gap_hours.dayofweek).map(ratio_by_dow).fillna(1.0).to_numpy()
    pred_counts = np.clip(np.rint(pred_counts * scale_hour * scale_dow), 0, max_count).astype(int)

# Optional: calibrate toward 710 pre-gap mean by (day_of_week, hour)
if (COUNT_BLEND_WEEKDAY is not None or COUNT_BLEND_WEEKEND is not None) and not p710_hist.empty:
    p710_hourly = p710_hist.copy()
    p710_hourly["hour_bucket"] = p710_hourly["typed_time"].dt.floor("h")
    p710_hourly_counts = p710_hourly.groupby("hour_bucket").size().reset_index(name="count")
    p710_hourly_counts["day_of_week"] = p710_hourly_counts["hour_bucket"].dt.dayofweek
    p710_hourly_counts["hour"] = p710_hourly_counts["hour_bucket"].dt.hour
    mean_by_hd = p710_hourly_counts.groupby(["day_of_week", "hour"])["count"].mean()
    mean_by_hd = mean_by_hd.reindex(pd.MultiIndex.from_product([range(7), range(24)]))
    gap_index = pd.MultiIndex.from_arrays([gap_hours.dayofweek, gap_hours.hour])
    expected = mean_by_hd.reindex(gap_index).to_numpy()
    overall_mean = p710_hourly_counts["count"].mean()
    expected = np.where(np.isnan(expected), overall_mean, expected)
    weekday_blend = 0.0 if COUNT_BLEND_WEEKDAY is None else COUNT_BLEND_WEEKDAY
    weekend_blend = 0.0 if COUNT_BLEND_WEEKEND is None else COUNT_BLEND_WEEKEND
    is_weekend = pd.Series(gap_hours.dayofweek).isin([5, 6]).to_numpy()
    blend = np.where(is_weekend, weekend_blend, weekday_blend)
    pred_counts = np.rint((1 - blend) * pred_counts + blend * expected)
    pred_counts = np.clip(pred_counts, 0, max_count).astype(int)

if ENFORCE_ACTIVE_HOURS and prob_active is not None:
    gap_index = pd.MultiIndex.from_arrays([gap_hours.dayofweek, gap_hours.hour])
    prob = prob_active.reindex(gap_index).to_numpy()
    prob_mean = float(np.nanmean(prob_active.values))
    prob = np.where(np.isnan(prob), prob_mean, prob)
    prob = np.clip(prob, ACTIVE_PROB_FLOOR, 1.0)
    active_mask = rng.random(len(gap_hours)) < prob
    pred_counts = (pred_counts * active_mask).astype(int)

if USE_710_WEEKDAY_SPIKES and p710_counts_by_dh is not None:
    is_weekend = pd.Series(gap_hours.dayofweek).isin([5, 6]).to_numpy()
    for i, (dow, hour) in enumerate(zip(gap_hours.dayofweek, gap_hours.hour)):
        if is_weekend[i] or pred_counts[i] <= 0:
            continue
        vals = p710_counts_by_dh.get((dow, hour))
        if not vals:
            continue
        spike_vals = [v for v in vals if v >= WEEKDAY_SPIKE_MIN]
        if spike_vals:
            vals = spike_vals
        spike = int(rng.choice(vals))
        if WEEKDAY_SPIKE_TAIL_POWER is None or len(vals) == 1:
            accept_prob = WEEKDAY_SPIKE_PROB
        else:
            vals_arr = np.asarray(vals)
            tail_prob = 1.0 - (vals_arr <= spike).mean()
            tail_prob = max(1.0 / len(vals_arr), tail_prob)
            accept_prob = WEEKDAY_SPIKE_PROB * (tail_prob ** WEEKDAY_SPIKE_TAIL_POWER)
        if rng.random() > accept_prob:
            continue
        if spike > pred_counts[i]:
            pred_counts[i] = min(spike, max_spike)

# Sample donor rows to create synthetic remissions
pool["day_of_week"] = pool["typed_time"].dt.dayofweek
pool["month"] = pool["typed_time"].dt.month

idx_full = pool.groupby(["hour", "day_of_week", "month"]).indices
idx_hour = pool.groupby(["hour"]).indices

sampled_idx = []
for hour_ts, n in zip(gap_hours, pred_counts):
    if n == 0:
        continue
    key = (hour_ts.hour, hour_ts.dayofweek, hour_ts.month)
    idxs = idx_full.get(key)
    if idxs is None:
        idxs = idx_hour.get(hour_ts.hour)
        if idxs is None:
            idxs = np.arange(len(pool))
    idxs = np.asarray(idxs)
    choice = rng.choice(idxs, size=n, replace=True)
    sampled_idx.append(choice)

if not sampled_idx:
    raise ValueError("No synthetic rows created. Check donor data or model predictions.")

sampled_idx = np.concatenate(sampled_idx)
synthetic = pool.iloc[sampled_idx].copy().reset_index(drop=True)

# Target hours for synthetic rows
rep_hours = np.repeat(gap_hours.values, pred_counts)
rep_hours = pd.to_datetime(rep_hours)

# Optional: use real 515 volume distribution
if USE_515_FOR_VOLUME and "u_Volumen" in synthetic.columns:
    vol_pool = p515_hist.dropna(subset=["typed_time", "u_Volumen"]).copy()
    if not vol_pool.empty:
        vol_pool["hour"] = vol_pool["typed_time"].dt.hour
        vol_pool["day_of_week"] = vol_pool["typed_time"].dt.dayofweek
        vol_pool["month"] = vol_pool["typed_time"].dt.month
        vol_idx_full = vol_pool.groupby(["hour", "day_of_week", "month"]).indices
        vol_idx_hour = vol_pool.groupby(["hour"]).indices
        vol_vals = []
        for ts in rep_hours:
            key = (ts.hour, ts.dayofweek, ts.month)
            idxs = vol_idx_full.get(key)
            if idxs is None:
                idxs = vol_idx_hour.get(ts.hour)
                if idxs is None:
                    idxs = np.arange(len(vol_pool))
            idx = rng.choice(idxs)
            vol_vals.append(vol_pool.iloc[idx]["u_Volumen"])
        synthetic["u_Volumen"] = vol_vals

# Preserve minute/second offsets from donor rows
for col in ["typed_time", "start_time", "at_plant_time"]:
    if col in synthetic.columns:
        src = pd.to_datetime(synthetic[col], errors="coerce")
        offset = src - src.dt.floor("h")
        synthetic[col] = rep_hours + offset.fillna(pd.Timedelta(0))

synthetic["order_date"] = rep_hours.normalize()

# Keep donor plant as metadata, overwrite target plant
synthetic["imputed_source_plant"] = synthetic["ship_plant_code"]
synthetic["ship_plant_code"] = PLANT_TARGET
synthetic["is_imputed"] = True
synthetic["imputed_method"] = "donor+rf"
synthetic["imputed_range"] = f"{GAP_START} to {GAP_END}"

if "year" in synthetic.columns:
    synthetic["year"] = synthetic["order_date"].dt.year

# Clear identifier-like codes except plant code
for col in remissions.columns:
    if col != "ship_plant_code" and col.lower().endswith("_code"):
        synthetic[col] = pd.NA

# Impute numeric columns with RF using donor pool
numeric_cols = remissions.select_dtypes(include=["number"]).columns.tolist()
exclude = [c for c in numeric_cols if "code" in c.lower() or c.lower().endswith("_id")]
exclude.append("year")
numeric_cols = [c for c in numeric_cols if c not in exclude]
if EXCLUDE_VOLUME_FROM_RF and "u_Volumen" in numeric_cols:
    numeric_cols.remove("u_Volumen")

if numeric_cols:
    train_mask = pool[numeric_cols].notna().all(axis=1)
    train_mask &= pool["typed_time"].notna()

    X_train = make_time_features(pool.loc[train_mask, "typed_time"])
    X_train["plant_code"] = pool.loc[train_mask, "ship_plant_code"].astype(str)
    X_train = pd.get_dummies(X_train, columns=["plant_code"], drop_first=False)

    y_train = pool.loc[train_mask, numeric_cols]

    rf_num = RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    model = MultiOutputRegressor(rf_num)
    model.fit(X_train, y_train)

    X_pred = make_time_features(synthetic["typed_time"])
    X_pred["plant_code"] = synthetic["imputed_source_plant"].astype(str)
    X_pred = pd.get_dummies(X_pred, columns=["plant_code"], drop_first=False)
    X_pred = X_pred.reindex(columns=X_train.columns, fill_value=0)

    int_cols = [c for c in numeric_cols if pd.api.types.is_integer_dtype(remissions[c])]
    synthetic[numeric_cols] = synthetic[numeric_cols].astype("Float64")

    y_pred = model.predict(X_pred)
    pred_df = pd.DataFrame(y_pred, columns=numeric_cols, index=synthetic.index)
    synthetic.loc[:, numeric_cols] = pred_df

    if APPLY_VOLUME_DOW_SCALE and ratio_515_volume is not None and "u_Volumen" in synthetic.columns:
        dow_factor = synthetic["typed_time"].dt.dayofweek.map(ratio_515_volume).fillna(1.0)
        dow_factor = dow_factor.clip(VOLUME_DOW_CLIP[0], VOLUME_DOW_CLIP[1])
        if VOLUME_DOW_BLEND is not None:
            dow_factor = (1 - VOLUME_DOW_BLEND) + VOLUME_DOW_BLEND * dow_factor
        synthetic["u_Volumen"] = synthetic["u_Volumen"] * dow_factor

    if not p710_hist.empty:
        q_low = p710_hist[numeric_cols].quantile(0.01)
        q_high = p710_hist[numeric_cols].quantile(0.99)
    else:
        q_low = pd.Series(index=numeric_cols, dtype=float)
        q_high = pd.Series(index=numeric_cols, dtype=float)

    donor_low = pool[numeric_cols].quantile(0.01)
    donor_high = pool[numeric_cols].quantile(0.99)
    q_low = q_low.fillna(donor_low)
    q_high = q_high.fillna(donor_high)

    for col in numeric_cols:
        synthetic[col] = synthetic[col].clip(q_low[col], q_high[col])

    if "u_Volumen" in synthetic.columns:
        synthetic["u_Volumen"] = synthetic["u_Volumen"].clip(lower=0.1)
    if "u_Cicle" in synthetic.columns:
        synthetic["u_Cicle"] = synthetic["u_Cicle"].clip(lower=1.0)

    if int_cols:
        synthetic[int_cols] = synthetic[int_cols].round().astype("Int64")

# Merge and export
imputed = pd.concat([remissions, synthetic], ignore_index=True)
imputed = imputed.sort_values("typed_time").reset_index(drop=True)

output_path = "../data/processed/remissions_db_imputed_710.xlsx"
imputed.to_excel(output_path, index=False)

print("Synthetic rows:", len(synthetic))
print("Total rows:", len(imputed))
print("Output:", output_path)

Donor plants: ['515', '512', '514', '511']
710 vs 515 day-of-week ratios (pre-gap):
             ratio_count_710_vs_515  ratio_volume_710_vs_515
day_of_week                                                 
0                          1.273359                 1.246449
1                          1.242764                 1.183465
2                          1.232574                 1.197072
3                          1.211877                 1.148183
4                          1.282900                 1.262361
5                          1.062741                 0.987581
6                          1.000000                 2.000000
Synthetic rows: 4699
Total rows: 358496
Output: ../data/processed/remissions_db_imputed_710.xlsx


In [2]:
# DATA IMPUTATION FOR ALL PLANTS MISSING 1 MONTH GAP
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

# General multioutput imputation for all plants using each plant's own history.
# This fills the missing period from 2025-01-01 through 2025-04-30.
DATA_PATH = "../data/processed/remissions_db_imputed_710.xlsx"
OUTPUT_PATH = "../data/processed/remissions_db_imputed_all_plants_2025.xlsx"
GAP_START = pd.Timestamp("2025-01-01")
GAP_END = pd.Timestamp("2025-05-01")
RANDOM_STATE = 42
MIN_HISTORY_HOURS = 48

def make_time_features(dt_series):
    dt = pd.to_datetime(dt_series)
    return pd.DataFrame({
        "hour": dt.dt.hour,
        "day_of_week": dt.dt.dayofweek,
        "month": dt.dt.month,
        "day_of_year": dt.dt.dayofyear,
        "is_weekend": (dt.dt.dayofweek >= 5).astype(int),
    })

remissions = pd.read_excel(DATA_PATH)
for col in ["order_date", "typed_time", "start_time", "at_plant_time"]:
    if col in remissions.columns:
        remissions[col] = pd.to_datetime(remissions[col], errors="coerce")

if "ship_plant_code" in remissions.columns:
    remissions["ship_plant_code"] = remissions["ship_plant_code"].astype(str).str.strip()

def impute_plant_gap(plant_code, plant_df):
    plant_df = plant_df.dropna(subset=["typed_time"]).copy()
    history = plant_df[plant_df["typed_time"] < GAP_START].copy()
    if history.empty or history["typed_time"].nunique() < MIN_HISTORY_HOURS:
        return None

    history["hour_bucket"] = history["typed_time"].dt.floor("h")
    hourly = (
        history.groupby("hour_bucket", as_index=False)
        .agg(
            remission_count=("typed_time", "size"),
            u_Volumen_sum=("u_Volumen", "sum"),
        )
    )
    hourly = hourly.dropna(subset=["remission_count", "u_Volumen_sum"])

    if len(hourly) < MIN_HISTORY_HOURS:
        return None

    history_active = history.copy()
    history_active["date"] = history_active["typed_time"].dt.normalize()
    history_active["hour"] = history_active["typed_time"].dt.hour
    history_active["day_of_week"] = history_active["typed_time"].dt.dayofweek
    days_by_dow = (
        history_active[["day_of_week", "date"]]
        .drop_duplicates()
        .groupby("day_of_week")
        .size()
    )
    active_by_dh = (
        history_active.groupby(["day_of_week", "hour", "date"]).size()
        .reset_index()
        .groupby(["day_of_week", "hour"]).size()
    )
    active_prob = (
        active_by_dh / days_by_dow
    ).reindex(pd.MultiIndex.from_product([range(7), range(24)]))
    active_prob = active_prob.fillna(0.0).clip(0.0, 1.0)

    X_train = make_time_features(hourly["hour_bucket"])
    y_train = hourly[["remission_count", "u_Volumen_sum"]].copy()
    y_train["u_Volumen_sum"] = y_train["u_Volumen_sum"].fillna(0)

    model = MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    )
    model.fit(X_train, y_train)

    full_gap_hours = pd.date_range(GAP_START, GAP_END - pd.Timedelta(hours=1), freq="h")
    existing_hours = plant_df["typed_time"].dt.floor("h").dropna().unique()
    gap_hours = full_gap_hours.difference(existing_hours)
    if gap_hours.empty:
        return None

    preds = model.predict(make_time_features(pd.Series(gap_hours)))
    pred_counts = np.clip(np.rint(preds[:, 0]), 0, None).astype(int)
    pred_volume_sum = np.clip(preds[:, 1], 0, None)

    gap_index = pd.MultiIndex.from_arrays([gap_hours.dayofweek, gap_hours.hour])
    gap_prob = active_prob.reindex(gap_index).to_numpy()
    prob_mean = float(np.nanmean(active_prob.values)) if np.isfinite(active_prob.values).any() else 0.0
    gap_prob = np.where(np.isnan(gap_prob), prob_mean, gap_prob)
    gap_prob = np.clip(gap_prob, 0.0, 1.0)

    rng = np.random.default_rng(RANDOM_STATE)
    active_mask = rng.random(len(gap_hours)) < gap_prob
    pred_counts = (pred_counts * active_mask).astype(int)

    donor_pool = history.copy()
    donor_pool["hour"] = donor_pool["typed_time"].dt.hour
    donor_pool["day_of_week"] = donor_pool["typed_time"].dt.dayofweek
    donor_pool["month"] = donor_pool["typed_time"].dt.month

    idx_full = donor_pool.groupby(["hour", "day_of_week", "month"]).indices
    idx_hour = donor_pool.groupby("hour").indices

    imputed_blocks = []

    for hour_ts, n_rows, target_volume in zip(gap_hours, pred_counts, pred_volume_sum):
        if n_rows <= 0:
            continue

        key = (hour_ts.hour, hour_ts.dayofweek, hour_ts.month)
        idxs = idx_full.get(key)
        if idxs is None:
            idxs = idx_hour.get(hour_ts.hour)
            if idxs is None:
                idxs = np.arange(len(donor_pool))

        sampled = donor_pool.iloc[rng.choice(np.asarray(idxs), size=n_rows, replace=True)].copy()

        for col in ["typed_time", "start_time", "at_plant_time"]:
            if col in sampled.columns:
                src = pd.to_datetime(sampled[col], errors="coerce")
                offset = src - src.dt.floor("h")
                sampled[col] = hour_ts + offset.fillna(pd.Timedelta(0))

        sampled["order_date"] = hour_ts.normalize()
        if "u_Volumen" in sampled.columns:
            current_volume = sampled["u_Volumen"].sum()
            if pd.notna(target_volume) and current_volume > 0:
                sampled["u_Volumen"] = sampled["u_Volumen"] * (target_volume / current_volume)
            sampled["u_Volumen"] = sampled["u_Volumen"].clip(lower=0.1)

        sampled["ship_plant_code"] = plant_code
        sampled["is_imputed"] = True
        sampled["imputed_method"] = "plant_history_multioutput"
        sampled["imputed_range"] = f"{GAP_START.date()} to {(GAP_END - pd.Timedelta(days=1)).date()}"
        sampled["source_plant_code"] = plant_code

        for col in sampled.columns:
            if col != "ship_plant_code" and col.lower().endswith("_code"):
                sampled[col] = pd.NA

        imputed_blocks.append(sampled)

    if not imputed_blocks:
        return None

    return pd.concat(imputed_blocks, ignore_index=True)

plant_codes = sorted(remissions["ship_plant_code"].dropna().unique())
all_imputed_blocks = []
imputation_log = []

for plant_code in plant_codes:
    plant_df = remissions[remissions["ship_plant_code"] == plant_code].copy()
    synthetic = impute_plant_gap(plant_code, plant_df)
    if synthetic is None:
        imputation_log.append({
            "ship_plant_code": plant_code,
            "status": "skipped",
            "rows_created": 0,
        })
        continue

    all_imputed_blocks.append(synthetic)
    imputation_log.append({
        "ship_plant_code": plant_code,
        "status": "imputed",
        "rows_created": len(synthetic),
    })

if all_imputed_blocks:
    imputed_all_plants = pd.concat([remissions] + all_imputed_blocks, ignore_index=True)
    imputed_all_plants = imputed_all_plants.sort_values("typed_time").reset_index(drop=True)
else:
    imputed_all_plants = remissions.copy()

imputed_all_plants.to_excel(OUTPUT_PATH, index=False)

print(imputation_log)
print("Original rows:", len(remissions))
print("Synthetic rows:", sum(item["rows_created"] for item in imputation_log))
print("Final rows:", len(imputed_all_plants))
print("Output:", OUTPUT_PATH)

[{'ship_plant_code': '510', 'status': 'imputed', 'rows_created': 1726}, {'ship_plant_code': '511', 'status': 'imputed', 'rows_created': 1680}, {'ship_plant_code': '512', 'status': 'imputed', 'rows_created': 2328}, {'ship_plant_code': '514', 'status': 'imputed', 'rows_created': 1708}, {'ship_plant_code': '515', 'status': 'imputed', 'rows_created': 1109}, {'ship_plant_code': '710', 'status': 'imputed', 'rows_created': 1387}]
Original rows: 358496
Synthetic rows: 9938
Final rows: 368434
Output: ../data/processed/remissions_db_imputed_all_plants_2025.xlsx


#### Data Imputation Check: Visualization of results to make tweaks and refinements

In [3]:
# Load and visualize the imputed dataset
import pandas as pd
import plotly.express as px

remissions_imputated = pd.read_excel("../data/processed/remissions_db_imputed_all_plants_2025.xlsx")
remissions_imputated.head(-1)

,tkt_code,order_date,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,...,map_page,hour,day_of_week,hour_bucket,month,imputed_source_plant,is_imputed,imputed_method,imputed_range,source_plant_code
0,51253281.0,2020-02-04,2020-02-04 05:00:00,7043.0,512,6.0,2020-02-04 05:30:00,2020-02-04 06:50:00,80,COPACHISA,...,CH-F3,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
1,51253282.0,2020-02-04,2020-02-04 05:00:00,4366.0,512,6.0,2020-02-04 05:31:00,2020-02-04 06:44:00,73,COPACHISA,...,CH-F3,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
2,51165719.0,2020-02-04,2020-02-04 07:00:00,6600.0,511,4.5,2020-02-04 06:23:00,2020-02-04 08:12:00,109,PARCELAS CHUVISCAR,...,CH-V1,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
3,51165721.0,2020-02-04,2020-02-04 07:00:00,10149.0,511,3.5,2020-02-04 06:24:00,2020-02-04 07:54:00,90,FERRETERIA MOLINA DE CHIHUAHUA,...,CH-N17,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
4,51165722.0,2020-02-04,2020-02-04 07:15:00,6608.0,511,3.0,2020-02-04 06:28:00,2020-02-04 07:53:00,85,ALVHER CORPORATIVO SA DE CV,...,CH-V3,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
368428,51168725.0,2026-04-24,2026-04-24 10:00:00,9636.0,511,5.0,2026-04-24 09:29:16,2026-04-24 10:32:06,63,INMOBILIARIA MERCIA 2019,...,CHN-17,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
368429,51025159.0,2026-04-24,2026-04-24 09:30:00,9431.0,510,1.0,2026-04-24 09:40:42,2026-04-24 10:28:21,48,PREMEZCLADOS Y MATERIALES PARA,...,CHH-1,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
368430,71038980.0,2026-04-24,2026-04-24 10:15:00,13041.0,710,5.5,2026-04-24 09:48:24,2026-04-24 10:45:11,57,DISEÑOS Y CONSTRUCCIONES CIVILES,...,CHN-2,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
368431,51428785.0,2026-04-24,2026-04-24 10:30:00,13008.0,514,3.0,2026-04-24 09:56:47,2026-04-24 10:47:43,51,RUBA DESARROLLOS,...,CHV-1,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# New size after imputation
print("Number of rows in remissions DataFrame:", remissions_imputated.shape[0])
print("Number of columns in remissions DataFrame:", remissions_imputated.shape[1])

Number of rows in remissions DataFrame: 368434
Number of columns in remissions DataFrame: 22


In [5]:
# Drop unused (helper) columns for visualization

remissions_imputated = remissions_imputated.drop(columns=["source_plant_code"], errors="ignore")

In [10]:
remissions_imputated = remissions_imputated.drop(columns=["hour", "day_of_week", "hour_bucket","month", "truck_code","imputed_source_plant","imputed_range","imputed_method" ], errors="ignore")

In [11]:
# See amount of remissions synthetic vs real
remissions_imputated["is_imputed"] = remissions_imputated["is_imputed"].fillna(False)
synthetic_count = remissions_imputated["is_imputed"].sum()
real_count = len(remissions_imputated) - synthetic_count
print(f"Synthetic remissions: {synthetic_count}")
print(f"Real remissions: {real_count}")
# show percentages
synthetic_percentage = (synthetic_count / len(remissions_imputated)) * 100
real_percentage = (real_count / len(remissions_imputated)) * 100
print(f"Synthetic remissions percentage: {synthetic_percentage:.2f}%")
print(f"Real remissions percentage: {real_percentage:.2f}%")

Synthetic remissions: 14637.0
Real remissions: 353797.0
Synthetic remissions percentage: 3.97%
Real remissions percentage: 96.03%


In [12]:
# print for plant 512 how many remissions in total per weekday for all time range
plant_512_data = remissions_imputated[remissions_imputated['ship_plant_code'] == 512]
plant_512_data['weekday'] = plant_512_data['typed_time'].dt.day_name()
remissions_per_weekday = plant_512_data.groupby('weekday').size().reset_index(name='remission_count')
print(remissions_per_weekday)
# Make it a histogram of weekdays (Monday -> Sunday)
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
remissions_per_weekday["weekday"] = pd.Categorical(
    remissions_per_weekday["weekday"],
    categories=weekday_order,
    ordered=True
)
remissions_per_weekday = remissions_per_weekday.sort_values("weekday")

fig = px.bar(
    remissions_per_weekday,
    x="weekday",
    y="remission_count",
    title="Remissions per Weekday for Plant 512",
    category_orders={"weekday": weekday_order}
)
fig.show()

     weekday  remission_count
0     Friday            17527
1     Monday            12234
2   Saturday             9478
3     Sunday              698
4   Thursday            17087
5    Tuesday            14094
6  Wednesday            13901


In [14]:
# Analyze hourly distribution of u_Volumen by ship_plant_code
hourly_distribution = (
    remissions_imputated
    .assign(hour_bucket=remissions_imputated["typed_time"].dt.floor("h"))
    .groupby(["hour_bucket", "ship_plant_code"], as_index=False)["u_Volumen"]
    .sum()
)

fig = px.line(
    hourly_distribution,
    x="hour_bucket",
    y="u_Volumen",
    color="ship_plant_code",
    title="Hourly Distribution of u_Volumen by Ship Plant Code (2020-2026)"
)
fig.show()

In [9]:
# Analyze hourly distribution of remissions (row_count) by ship_plant_code (meaning rounding typed_time to the hour and counting rows)
hourly_distribution = (
    remissions_imputated
    .assign(hour_bucket=remissions_imputated["typed_time"].dt.floor("h"))
    .groupby(["hour_bucket", "ship_plant_code"], as_index=False)
    .size()
    .rename(columns={"size": "remission_count"})
)

hourly_distribution["u_Volumen"] = hourly_distribution["remission_count"]

fig = px.line(
    hourly_distribution,
    x="hour_bucket",
    y="remission_count",
    color="ship_plant_code",
    title="Hourly Distribution of Remissions by Ship Plant Code (2020-2026)"
)
fig.show()